# **Tech Challenge - Fase 3**<br>
Este projeto implementa um ecossistema de assistência médica personalizada, integrando três pilares fundamentais da Inteligência Artificial moderna:

1.   **Fine-Tuning (Unsloth):** Otimização do modelo Llama-3 para terminologia médica e protocolos específicos.
2.   **RAG (Retrieval-Augmented Generation):** Recuperação de contextos em tempo real via ChromaDB para reduzir alucinações.
3. **Orquestração de Fluxo (LangGraph):** Gerenciamento de estado e controle cíclico de decisões entre os agentes de pesquisa e geração.


## **Download e processamento dos dados**

Nesta etapa, realizamos o pipeline de ingestão de dados estruturados e não-estruturados. O foco principal é a segurança da informação (LGPD):

* **Anonimização:** Remoção de identificadores sensíveis (PII) dos registros de pacientes.

* **Vetorização:** Fragmentação (chunking) de protocolos médicos e armazenamento em base vetorial para busca semântica.

* **Sintetização:** Preparação do dataset no formato Alpaca para o ajuste fino do modelo.

In [1]:
# Base
!pip install -q pandas pyarrow requests

# LLM / Fine-tuning
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q transformers datasets peft accelerate bitsandbytes "trl>=0.18.2,<0.25.0"

# LangChain + RAG
!pip install -q langchain langchain-community langchain-huggingface langgraph
!pip install -q chromadb sentence-transformers InstructorEmbedding
!pip install -q langchain langchain-community chromadb sentence-transformers InstructorEmbedding
!pip install -q -U langchain-text-splitters
!pip install -q -U langchain-huggingface
!pip install -q -U huggingface_hub sentence-transformers

# Utilitários extras (opcional)
!pip install -q newspaper3k html2text




  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import logging

# Configuração do Logger para arquivo e console
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("assistente_medico_audit.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)
logger.info("Sistema de Assistente Médico Iniciado.")

In [4]:
from datasets import load_dataset

dataset = load_dataset("AKCIT/MedPT", split="train", streaming=False)
#https://huggingface.co/datasets/AKCIT/MedPT/viewer/default/train?p=3840
logger.info("="*50)
logger.info("Dataset carregado com sucesso!")
logger.info("="*50)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
import pandas as pd
import pyarrow as pa

df = pd.DataFrame(dataset)
df.head()

,id,question,answer,condition,medical_specialty,question_type
0,0,Fazer aplicação de ácido hialurônico no ombro ...,"Pode funcionar,associado com fisioterapia.",Capsulite adesiva,Ortopedista - traumatologista,Tratamento
1,1,Fazer aplicação de ácido hialurônico no ombro ...,O bloqueio do movimento causado pela Capsulite...,Capsulite adesiva,Ortopedista - traumatologista,Tratamento
2,2,Posso fazer uso regularmente de alupurinol dia...,"Olá! Sim, pode ser necessário utilizar medicaç...",Artrite Gotosa,"Reumatologista, Médico acupunturista",Tratamento
3,4,O que é artrite gotosa?o que provoca isto?,Artrite gotosa é A inflamação de uma articulaç...,Artrite Gotosa,"Reumatologista, Especialista em dor, Internista",Diagnóstico
4,5,O que é artrite gotosa?o que provoca isto?,A artrite gotosa é uma inflamação dentro da ar...,Artrite Gotosa,Reumatologista,Diagnóstico


In [6]:
import re
import pandas as pd

class PreProcessingData:
    def __init__(self, dataset_name="AKCIT/MedPT"):
        self.dataset_name = dataset_name
        self.patterns = {
            'NOME': r'\b[A-Z][a-z]+(?:\s[A-Z][a-z]+)+\b',
            'CPF': r'\d{3}\.\d{3}\.\d{3}-\d{2}',
            'CRM': r'CRM-[A-Z]{2}\s*\d+',
            'PHONE': r'\(\d{2}\)\s\d{4,5}-\d{4}',
            'EMAIL': r'[\w\.-]+@[\w\.-]+\.[\w]+',
            'DATA': r'\b\d{2}/\d{2}/\d{4}\b',
        }

    def load_and_clean(self, df):
        df = df.drop(columns=['id', 'question_type', 'medical_speciality'], errors='ignore')
        df = df.drop_duplicates().dropna()

        cols_to_fix = ['question', 'answer']
        for col in cols_to_fix:
            if col in df.columns:
                df[col] = df[col].astype(str).str.lower().str.strip()

        logger.info("Dataset limpo com sucesso!")
        return df

    def anonymize_text(self, text):
        if not isinstance(text, str):
            return text

        for label, pattern in self.patterns.items():
            text = re.sub(pattern, f"[{label}_REMOVIDO]", text)

        return text

    def anonymize_record(self, record):
        anonymized = {}
        for key, value in record.items():
            if isinstance(value, str):
                anonymized[key] = self.anonymize_text(value)
            elif isinstance(value, list):
                anonymized[key] = [self.anonymize_text(v) for v in value]
            else:
                anonymized[key] = value
        return anonymized

    def clean_dataframe(self, df):
        text_columns = df.select_dtypes(include=["object"]).columns

        for col in text_columns:
            df[col] = df[col].apply(lambda x: self.anonymize_text(x))

        logger.info(f"Dataset anonimizado: {len(df)} registros")
        return df

record = {
    "paciente": "Felipe Mansouto",
    "cpf": "123.456.789-00",
    "data_nascimento": "23/06/1975",
    "exame": "Hemograma Completo",
    "medico": "Dra. Marica Menezes Silva",
    "crm": "CRM-SP 987654"
}

anonymizer = PreProcessingData()
record_anonymized = anonymizer.anonymize_record(record)
print(record_anonymized)

{'paciente': '[NOME_REMOVIDO]', 'cpf': '[CPF_REMOVIDO]', 'data_nascimento': '[DATA_REMOVIDO]', 'exame': '[NOME_REMOVIDO]', 'medico': 'Dra. [NOME_REMOVIDO]', 'crm': '[CRM_REMOVIDO]'}


In [7]:
logger.info("Iniciando limpeza do dataset!")
logger.info("="*50)

df = anonymizer.load_and_clean(df)
df = anonymizer.clean_dataframe(df)

df.head()

,question,answer,condition,medical_specialty
0,fazer aplicação de ácido hialurônico no ombro ...,"pode funcionar,associado com fisioterapia.",Capsulite adesiva,Ortopedista - traumatologista
1,fazer aplicação de ácido hialurônico no ombro ...,o bloqueio do movimento causado pela capsulite...,Capsulite adesiva,Ortopedista - traumatologista
2,posso fazer uso regularmente de alupurinol dia...,"olá! sim, pode ser necessário utilizar medicaç...",[NOME_REMOVIDO],"Reumatologista, Médico acupunturista"
3,o que é artrite gotosa?o que provoca isto?,artrite gotosa é a inflamação de uma articulaç...,[NOME_REMOVIDO],"Reumatologista, Especialista em dor, Internista"
4,o que é artrite gotosa?o que provoca isto?,a artrite gotosa é uma inflamação dentro da ar...,[NOME_REMOVIDO],Reumatologista


In [8]:
import os
import json

# Convertendo dados ficticios para instruções

CAMPO_INSTRUCAO = ["pergunta", "pergunta_medico", "protocolo", "exame", "tipo", "question"]
CAMPO_RESPOSTA = ["resposta", "descricao", "interpretacao", "modelo", "answer"]

def generate_pairs(data):
    pairs = []
    for registers in data.values():
        if not isinstance(registers, list):
            continue

        for r in registers:
            if not isinstance(r, dict):
                continue

            instr = next((r[c] for c in CAMPO_INSTRUCAO if c in r and r[c]), None)
            resp = next((r[c] for c in CAMPO_RESPOSTA if c in r and r[c]), None)

            if instr and resp and len(instr) > 5:
                pairs.append({
                    "instruction": str(instr),
                    "response": str(resp)
                })

    return pairs

import json

def save_llama(pairs, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        for par in pairs:
            item = {
                "instruction": par["instruction"],
                "input": "",  # campo vazio se não houver contexto
                "output": par["response"]
            }
            f.write(json.dumps(item, ensure_ascii=False) + "\n")


def load_jsons(folder_path):
    todos_dados = {}

    for arquivo in os.listdir(folder_path):
        if arquivo.endswith(".json"):
            caminho = os.path.join(folder_path, arquivo)
            with open(caminho, "r", encoding="utf-8") as f:
                todos_dados[arquivo] = json.load(f)

    return todos_dados


def anonymize_all(data, anonymizer):
    data_anon = {}

    for nome_arq, registros in data.items():
        if not isinstance(registros, list):
            continue

        data_anon[nome_arq] = [
            anonymizer.anonymize_record(r)
            for r in registros if isinstance(r, dict)
        ]

    return data_anon

In [9]:
folder_path = "/content/tech-challenge-fase3/fictitious_data"
output_path = "/content/tech-challenge-fase3/treino_llama/treino_llama.json"
os.makedirs(folder_path, exist_ok=True)
os.makedirs(os.path.dirname(output_path), exist_ok=True)

In [10]:
logger.info("Iniciando limpeza dos jsons!")
logger.info("="*50)

print("Estrutura de pastas verificada e criada!")

dados_json = load_jsons(folder_path)
dados_anon = anonymize_all(dados_json, anonymizer)
pares = generate_pairs(dados_anon)

logger.info(f"{len(pares)} pares gerados para treino")

save_llama(pares, output_path)

logger.info("Dataset final para LLaMA salvo com sucesso!")
logger.info("="*50)

Estrutura de pastas verificada e criada!


In [11]:
import json

with open(output_path) as f:
    for i, line in enumerate(f):
      print(json.loads(line))
      if i == 3:
        break

{'instruction': 'Relatório de Série Vermelha e Branca', 'input': '', 'output': 'Análise Laboratorial: Eritrograma e Leucograma. Contagem de Hemácias (milhões/mm³), Dosagem de Hemoglobina (g/dL) e [NOME_REMOVIDO] (%). Índices Hematimétricos: VCM, HCM e CHCM. [NOME_REMOVIDO] de Leucócitos e Diferencial (Neutrófilos, Linfócitos, Monócitos). Séries Plaquetária. Impressão Diagnóstica: [texto]. Nota: Resultado pendente de correlação clínica. Assinatura eletrônica do responsável.'}
{'instruction': 'Resultado de Glicose Plasmática', 'input': '', 'output': 'Exame: Glicemia de Jejum. Valor apurado: X mg/dL. Parâmetro de Normalidade: 70 a 99 mg/dL. Conclusão: [dentro dos limites / acima do esperado]. Observação: Resultados alterados devem ser confirmados com novos testes e avaliação especializada. Identificação da unidade e data.'}
{'instruction': 'Prescrição de [NOME_REMOVIDO]', 'input': '', 'output': 'Receituário Médico. Identificação do Paciente: [nome]. Fármaco: [princípio ativo], [miligramag

## **Fine-Tuning com Quantização 4-bit (QLoRA)**

Para viabilizar o uso de modelos de larga escala (LLMs) em hardware otimizado, utilizamos a biblioteca Unsloth. Esta abordagem permite:

* **Eficiência de Memória:** Redução do consumo de VRAM sem perda significativa de acurácia.

* **Aprendizado Especializado:** O modelo é treinado para interpretar condutas médicas e sugerir diagnósticos diferenciais baseados nos documentos internos fornecidos.

In [12]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch

from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

dataset = Dataset.from_pandas(df)

OUTPUT_PATH_DATASET = "/content/drive/MyDrive/Colab Notebooks/Fase 3 - FIAP/TECH CHALLENGE 3"

max_seq_length = 1024
 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.
# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
]

logger.info("Carregando modelo base...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Phi-3.5-mini-instruct", # Changed model to a smaller one
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    device_map = 'auto' # Added device_map='auto' to handle GPU memory
)

logger.info("Modelo carregado com sucesso!")
logger.info("="*50)

logger.info("Aplicando configuração LoRA (PEFT)...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",

    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

logger.info("LoRA aplicada com sucesso!")
logger.info("="*50)

logger.info("Iniciando preparação dos prompts...")

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
# Cria a coluna "text" no formato Alpaca
def formatting_prompts_func(examples):
    texts = []
    for q, c, a in zip(examples["question"], examples["condition"], examples["answer"]):
        input_text = f"Condição do paciente: {c}" if c else ""
        text = alpaca_prompt.format(q, input_text, a) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)


logger.info("="*50)
logger.info("Inicializando SFTTrainer...")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    formatting_func = formatting_prompts_func, # Adicionado o parâmetro formatting_func
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    )
)

logger.info("Trainer configurado!")
logger.info("="*50)

logger.info("Iniciando treinamento...")

trainer_stat = trainer.train()

logger.info("Treinamento finalizado!")
logger.info(f"Resultado: {trainer_stat}")

logger.info("="*50)

logger.info("Iniciando modo de inferência...")

# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

logger.info("Gerando resposta do modelo...")
logger.info("Inferência concluída!")
logger.info("="*50)

prompt = alpaca_prompt.format(
        "Using the information from the questions and answers in the data, provide an answer to the user. The answer should be analyzed based on the symptoms.", # instruction
        "", # input
        "", # output - leave this blank for generation!
  )
inputs = tokenizer(
    prompt,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=64,
    use_cache=False,
    do_sample=True,
    temperature=0.7,
    pad_token_id=tokenizer.eos_token_id # Adicionado para melhor tratamento de padding
    )

tokenizer.batch_decode(outputs)

logger.info(f"Salvando modelo em: {OUTPUT_PATH_DATASET}")
model.save_pretrained(OUTPUT_PATH_DATASET) # Local saving
tokenizer.save_pretrained(OUTPUT_PATH_DATASET)
logger.info("Modelo salvo com sucesso!")
logger.info("="*50)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.15: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth 2026.3.15 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Map:   0%|          | 0/384088 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/384088 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 32009}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 384,088 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,884,416 of 3,850,963,968 (0.78% trained)


Step,Training Loss
1,2.607781
2,2.618477
3,2.453006
4,2.532937
5,2.441321
6,2.239987
7,2.091237
8,2.172718
9,1.940711
10,2.022786


Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


In [13]:
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import json
import shutil # Importado para remover o diretório
import os # Importado para manipulação de caminhos
import time # Importado para adicionar atraso

from langchain_huggingface import HuggingFaceEmbeddings

# Substituindo o Instructor-XL pelo HuggingFaceEmbeddings padrão
# Este modelo não requer patches e é compatível com Python 3.12+
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

logger.info("Embeddings configurados com sucesso utilizando HuggingFaceEmbeddings.")


docs = []

for nome_arq, lista_registros in dados_anon.items():
    for r in lista_registros:
        instr = next((r[c] for c in CAMPO_INSTRUCAO if c in r and r[c]), "")
        resp = next((r[c] for c in CAMPO_RESPOSTA if c in r and r[c]), "")

        conteudo_limpo = f"{instr} {resp}".strip()

        if len(conteudo_limpo) > 10:
            doc = Document(
                page_content=conteudo_limpo,
                metadata={"fonte": nome_arq}
            )
            docs.append(doc)

print(f"Sucesso! {len(docs)} documentos clínicos modelados e prontos.")

# Splitting
splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " "]
)
chunks = splitter.split_documents(docs)
contents = [chunk.page_content for chunk in chunks]
sources = [chunk.metadata for chunk in chunks]

# Persistência do Chroma
persist_directory = '/content/tech-challenge-fase3/db/db_medical'
base_db_directory = '/content/tech-challenge-fase3/db'

# Certifica-se de que o diretório pai existe e é gravável
os.makedirs(os.path.dirname(persist_directory), exist_ok=True)

vectordb = Chroma.from_texts(contents, embedding, metadatas=sources, persist_directory=persist_directory)
vectordb.persist()

# Carregar a DB para uso
vectordb = Chroma(embedding_function=embedding, persist_directory=persist_directory)

# Recuperação
retriever = vectordb.as_retriever(search_type="similarity", search_kwargs={"k":5})
query = input("Faça uma pergunta:")
context = retriever.invoke(query)
print(context)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sucesso! 35 documentos clínicos modelados e prontos.


/tmp/ipykernel_16090/3300117893.py:54: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()
/tmp/ipykernel_16090/3300117893.py:57: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb = Chroma(embedding_function=embedding, persist_directory=persist_directory)


Faça uma pergunta:O que é doença renal crônica?
[Document(metadata={'fonte': 'fictitious_doctor_questions.json'}, page_content='O que caracteriza a doença renal crônica? Trata-se do declínio gradual e permanente da capacidade filtrante dos rins. As metas do tratamento são frear a evolução da doença através do controle da glicemia e pressão, além de ajustes dietéticos. Em fases críticas, pode haver indicação de métodos dialíticos ou transplante, sob orientação do nefrologista.'), Document(metadata={'fonte': 'fictitious_doctor_questions.json'}, page_content='O que caracteriza a doença renal crônica? Trata-se do declínio gradual e permanente da capacidade filtrante dos rins. As metas do tratamento são frear a evolução da doença através do controle da glicemia e pressão, além de ajustes dietéticos. Em fases críticas, pode haver indicação de métodos dialíticos ou transplante, sob orientação do nefrologista.'), Document(metadata={'fonte': 'fictitious_patient_exam_data.json'}, page_content='C

## **Orquestração com LangGraph**
Diferente de fluxos lineares, o LangGraph permite que o assistente gerencie um Estado Médico (MedicalState).

* **ResearchNode:** Atua como um bibliotecário, buscando evidências científicas no ChromaDB.

* **LLMNode:** Atua como o especialista clínico, sintetizando a resposta final integrando o conhecimento prévio (Fine-Tuned) e os dados recuperados (RAG).

In [23]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from unsloth import FastLanguageModel
import logging

# Recuperando o logger configurado anteriormente no notebook
logger = logging.getLogger(__name__)

class MedicalState(TypedDict):
    question: str
    context: str
    patient_data: str
    answer: str
    source: str

def obter_dados():
    """
    Extrai informações dos registros que foram anonimizados no início do processo.
    """
    pacientes_processados = []

    for nome_arquivo, registros in dados_anon.items():
        for r in registros:
            nome = r.get('paciente') or r.get('nome') or "Paciente"
            idade = r.get('idade') or r.get('data_nascimento') or "N/A"
            historico = r.get('exame') or r.get('descricao') or r.get('medicamento') or "Sem detalhes"

            pacientes_processados.append({
                'nome': nome,
                'idade': idade,
                'historico': historico[:100],
                'origem': nome_arquivo
            })

    # Retornamos apenas os 50 primeiros para não estourar a memória do Llama
    return pacientes_processados[:3]

# Nó de Recuperação de Dados (RAG)
def research_node(state: MedicalState):
    logger.info("--- INÍCIO: ResearchNode ---")
    query = state["question"]
    logger.info(f"Pergunta recebida: '{query}'")

    # Recuperação no ChromaDB
    docs = retriever.invoke(query)
    protocol_context = "\n".join(d.page_content for d in docs)
    logger.info(f"RAG: {len(docs)} documentos recuperados do ChromaDB.")

    # Processamento de dados de pacientes
    dados = obter_dados()
    patient_data_list = [
        f"[{p['origem']}] Paciente: {p['nome']}, {p['idade']}. Histórico: {p['historico']}"
        for p in dados
    ]
    patient_info = "\n".join(patient_data_list)

    state["context"] = protocol_context
    state["patient_data"] = patient_info
    state["source"] = "ChromaDB + Protocolos"

    logger.info("FIM: ResearchNode - Dados de contexto e pacientes anexados ao estado.")
    return state


# Nó de Geração LLM
def llm_node(state: MedicalState):
    logger.info("--- INÍCIO: LLMNode ---")

    prompt = alpaca_prompt.format(
        "Você é um assistente médico. Use o contexto e queixas do paciente para sugerir uma conduta. Termine com: Consulte um médico para validação.",
        f"Contexto: {state['context']}\nDados do Paciente: {state['patient_data']}\nPergunta: {state['question']}",
        ""
    )

    logger.info("Iniciando geração de resposta com Fine-Tuned LLM...")
    FastLanguageModel.for_inference(model)

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens = 150,
        use_cache = False,
        temperature=0.3)

    response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    final_answer = response.split("### Response:")[-1].strip()

    aviso = "\n\nIMPORTANTE: Esta é uma sugestão baseada em protocolos e deve ser validada por um médico profissional."

    if "consultar" not in final_answer.lower():
        final_answer += aviso
    state["answer"] = final_answer
    state["source"] += " + LLM Fine-Tuned"

    logger.info("FIM: LLMNode - Resposta gerada com sucesso.")
    return state


def inferencia_llm(prompt: str, modelo, tokenizer_model) -> str:
    tokenizer_model.padding_side = "left"
    try:
        if modelo is None or tokenizer_model is None:
            return "[Erro] Modelo não carregado."

        inputs = tokenizer_model([prompt], return_tensors="pt", padding=True, truncation=True, max_length=1024).to("cuda")

        outputs = modelo.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.2,
            do_sample=True,
            eos_token_id=tokenizer_model.eos_token_id
        )

        resposta = tokenizer_model.batch_decode(outputs, skip_special_tokens=True)[0]

        # Limpeza cirúrgica:
        if "<|assistant|>" in resposta:
            final_answer = resposta.split("<|assistant|>")[-1].strip()
        else:
            final_answer = resposta.strip()

        return final_answer
    except Exception as e:
        return f"Erro: {e}"



In [25]:
# Construção do grafo
workflow = StateGraph(MedicalState)

workflow.add_node("ResearchNode", research_node)
workflow.add_node("LLMNode", llm_node)

workflow.set_entry_point("ResearchNode")
workflow.add_edge("ResearchNode", "LLMNode")
workflow.add_edge("LLMNode", END)

# Compilação
app = workflow.compile()

# Execução com .invoke()
logger.info("Iniciando execução do Grafo de Atendimento Médico.")
inputs = {
    "question": "O que é artrite gotosa?",
    "context": "",
    "patient_data": "",
    "answer": "",
    "source": ""
}

final_state = app.invoke(inputs)

print("\n" + "="*50)
print("RESPOSTA FINAL:\n", final_state["answer"])
print("="*50)
logger.info("Fluxo finalizado com sucesso.")

Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



RESPOSTA FINAL:
 a artrite gotosa é uma doença inflamatória que afeta as articulações, principalmente as do joelho. a artrite gotosa é uma doença autoimune, ou seja, o próprio sistema imunológico do paciente ataca as articulações. a artrite gotosa é uma doença crônica, ou seja, não tem cura, mas pode ser controlada com tratamento. a artrite gotosa pode causar dor, inchaço e rigidez nas articulações, principalmente no joelho. a artrite gotosa pode afetar qualquer pessoa, mas é mais comum em

IMPORTANTE: Esta é uma sugestão baseada em protocolos e deve ser validada por um médico profissional.


In [22]:
from langgraph.graph import StateGraph, END
import logging

logger = logging.getLogger(__name__)

# 1. Definir o Grafo
workflow_test = StateGraph(MedicalState)

# 2. Adicionar os nós usando as funções corretas
workflow_test.add_node("ResearchNode", research_node)
workflow_test.add_node("LLMNode", llm_node)

# 3. Configurar o fluxo
workflow_test.set_entry_point("ResearchNode")
workflow_test.add_edge("ResearchNode", "LLMNode")
workflow_test.add_edge("LLMNode", END)

# 4. COMPILAR
app_test = workflow_test.compile()

# -------------------------
# Executar teste
# -------------------------
print("Executando teste final com proteções de inferência...\n")

input = {
    "question": "Existe problema em suspender o antibiótico se eu já estiver bem?",
    "context": "",
    "patient_data": "",
    "answer": "",
    "source": ""
}

resultado_final = app_test.invoke(input)

print("\n" + "="*50)
print("--- RESPOSTA FINAL DO ASSISTENTE ---")
print(resultado_final["answer"])
print("="*50)
print(f"Fonte dos dados: {resultado_final['source']}")

Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Executando teste final com proteções de inferência...


--- RESPOSTA FINAL DO ASSISTENTE ---
não. é imperativo completar o ciclo e as doses estabelecida na receita médica. a interrupção precoce favorece o surgimento de bactérias resistentes e o risco de recidiva da infecção. caso surjam reações indesejadas, relate ao médico antes de qualquer mudança.
Fonte dos dados: ChromaDB + Protocolos + LLM Fine-Tuned
